# PubMed: thử lại riêng các PMID còn thiếu

**Chạy một cell code bên dưới.** Cell đọc các PMID PubMed chưa ở trạng thái `ok` từ chính `crawl/status.duckdb`, thử NCBI EFetch theo batch 10 PMID, rồi thử từng PMID chưa có kết quả. Mọi XML và kết quả yêu cầu đều được lưu để có thể chạy tiếp. Nếu lấy thêm được bài, cell sao lưu và cập nhật `status.duckdb`, rồi tạo **một bộ docs Parquet mới**; bộ docs cũ vẫn còn nguyên.

Cell này chỉ xử lý PMID còn thiếu. Nếu NCBI vẫn không trả nội dung cho một PMID sau hai cách thử, nó sẽ tiếp tục được liệt kê trong báo cáo, không bị gắn nhầm là đã crawl thành công. Dừng các tiến trình khác đang ghi DuckDB trước khi chạy.


In [ ]:
# 1. Dependencies, Drive, and paths
print('PubMed only: preparing dependencies and Drive...', flush=True)
import importlib.util
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

requirements = {
    'duckdb': 'duckdb', 'httpx': 'httpx', 'pandas': 'pandas',
    'pyarrow': 'pyarrow', 'zstandard': 'zstandard',
}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('Installing:', ', '.join(missing), flush=True)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/r2ai_data')
CRAWL_DIR = DATA_ROOT / 'crawl'
STATUS_DB = CRAWL_DIR / 'status.duckdb'
PUBMED_DIR = CRAWL_DIR / 'pubmed'
DOCS_DIR = DATA_ROOT / 'docs'
REPORT_DIR = DATA_ROOT / 'reports' / (
    'pubmed_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
if not STATUS_DB.is_file():
    raise FileNotFoundError(f'Status DB not found: {STATUS_DB}')
batch_files = sorted(PUBMED_DIR.glob('batch_*.json.zst'))
if not batch_files:
    raise FileNotFoundError(f'PubMed cache batches not found: {PUBMED_DIR}')
print(f'Status DB: {STATUS_DB}', flush=True)
print(f'Europe PMC cache: {len(batch_files)} batch files in {PUBMED_DIR}', flush=True)
print(f'Docs output root: {DOCS_DIR}', flush=True)

# 2. PubMed-only schema-v2 cleaner
"""PubMed-only cleaner embedded in the standalone Colab notebook.

The output follows the same schema-v2 contract as notebook 03.
"""

import hashlib
import re
from datetime import datetime
from html import unescape as html_unescape
from typing import Any, Dict, Optional


_ZW_RE = re.compile(r'[\u200b\u2060\ufeff]')
SAFE_TAG_RE = re.compile(r'</?[A-Za-z][A-Za-z0-9]*(?:\s[^<>]*)?/?>')


def normalize_ws(text: str) -> str:
    if not text:
        return ''
    return re.sub(r'\s+', ' ', _ZW_RE.sub('', text)).strip()


def parse_published_date(value: Any) -> Optional[str]:
    if value is None:
        return None
    raw = normalize_ws(str(value))
    if not raw:
        return None
    for cand in (raw[:10], raw):
        for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%d/%m/%Y',
                    '%m/%d/%Y', '%B %d, %Y', '%b %d, %Y'):
            try:
                return datetime.strptime(cand, fmt).date().isoformat()
            except ValueError:
                pass
    match = re.fullmatch(r'(\d{4})年(\d{1,2})月(\d{1,2})日?', raw)
    if match:
        try:
            return datetime(int(match.group(1)), int(match.group(2)),
                            int(match.group(3))).date().isoformat()
        except ValueError:
            return None
    return None


def strip_html_tags_safe(text: str) -> str:
    return html_unescape(SAFE_TAG_RE.sub('', text))


def extract_pubmed_doc(pm_rec: Dict[str, Any], doc_id: int, url: str) -> Dict[str, Any]:
    title = normalize_ws(strip_html_tags_safe(pm_rec.get('title', '') or ''))
    if title.endswith('.'):
        title = title[:-1]
    blocks, text_parts = [], []
    char_cursor = 0

    def add_block(btype, text, level=None, role=None):
        nonlocal char_cursor
        cs, ce = char_cursor, char_cursor + len(text)
        char_cursor = ce + 2
        blocks.append({'type': btype, 'level': level, 'role': role, 'text': text,
                       'char_start': cs, 'char_end': ce})
        text_parts.append(text)

    if title:
        add_block('heading', title, level=1)
    abs_text = pm_rec.get('abstractText') or pm_rec.get('abstract') or ''
    if isinstance(abs_text, str) and abs_text:
        if '<' in abs_text:
            parts = re.split(r'<h4[^>]*>(.*?)</h4>', abs_text)
            p0 = normalize_ws(strip_html_tags_safe(parts[0]))
            if p0:
                add_block('paragraph', p0)
            for lbl_raw, cnt_raw in zip(parts[1::2], parts[2::2]):
                lbl = normalize_ws(strip_html_tags_safe(lbl_raw)).rstrip(':')
                cnt = normalize_ws(strip_html_tags_safe(cnt_raw))
                if cnt:
                    add_block('paragraph', f'{lbl}: {cnt}' if lbl else cnt)
        else:
            add_block('paragraph', normalize_ws(abs_text))
    keyword_list = pm_rec.get('keywordList') or {}
    kws = keyword_list.get('keyword', []) if isinstance(keyword_list, dict) else []
    if kws:
        add_block('paragraph', 'Keywords: ' + '; '.join(str(k) for k in kws) + '.')
    full_text = '\n\n'.join(text_parts)
    has_abstract = isinstance(abs_text, str) and bool(normalize_ws(strip_html_tags_safe(abs_text)))
    flags = [] if has_abstract else ['no_abstract']
    return {
        'id': doc_id, 'url': url, 'final_url': url,
        'lang': 'en', 'lang_conf': 1.0,
        'title': title, 'published_date': parse_published_date(pm_rec.get('firstPublicationDate')),
        'site': 'pubmed.ncbi.nlm.nih.gov',
        'category': ((pm_rec.get('journalInfo') or {}).get('journal') or {}).get('title'),
        'text': full_text, 'blocks': blocks, 'n_chars': len(full_text),
        'sha1_text': hashlib.sha1(full_text.encode('utf-8')).hexdigest(),
        'dup_group': None, 'extractor': 'pubmed_europepmc_v3',
        'quality_flags': flags, 'schema_version': 2,
    }

# 3. NCBI completion, DB sync, and Parquet writer
"""Complete and extract PubMed records in Colab.

This source is embedded in notebooks 03 and 04. Keep all copies in sync.
"""

import csv
import hashlib
import html
import io
import json
import os
import re
import shutil
import time
import uuid
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse

import duckdb
import httpx
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import zstandard as zstd


NCBI_EFETCH_URL = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi'
NCBI_BATCH_SIZE = 200
PUBMED_DOC_SHARD_SIZE = 10_000


def _pubmed_source_rows(status_db):
    con = duckdb.connect(str(status_db), read_only=True)
    try:
        rows = con.execute("""SELECT id, url, status FROM crawl_status
            WHERE lower(url) LIKE '%pubmed.ncbi.nlm.nih.gov%'""").fetchall()
    finally:
        con.close()
    source = {}
    for rid, url, status in rows:
        parsed = urlparse(str(url))
        pmid = parsed.path.strip('/')
        if parsed.hostname != 'pubmed.ncbi.nlm.nih.gov' or not pmid.isdigit():
            continue
        if pmid in source:
            raise ValueError(f'Duplicate PMID in crawl_status: {pmid}')
        source[pmid] = (int(rid), str(url), str(status or ''))
    if not source:
        raise ValueError('No PubMed PMID URLs in crawl_status')
    return source


def _xml_text(node):
    return ''.join(node.itertext()).strip() if node is not None else ''


def _ncbi_date(article):
    for base in ('./MedlineCitation/Article/ArticleDate',
                 './MedlineCitation/Article/Journal/JournalIssue/PubDate',
                 './BookDocument/ArticleDate'):
        date = article.find(base)
        if date is None:
            continue
        year, month, day = (date.findtext(name) for name in ('Year', 'Month', 'Day'))
        if not (year and month and day):
            continue
        if not month.isdigit():
            for fmt in ('%b', '%B'):
                try:
                    month = str(datetime.strptime(month, fmt).month)
                    break
                except ValueError:
                    pass
        try:
            return datetime(int(year), int(month), int(day)).date().isoformat()
        except (TypeError, ValueError):
            continue
    return None


def _parse_ncbi_xml(raw):
    root = ET.fromstring(raw)
    if root.tag.rsplit('}', 1)[-1] != 'PubmedArticleSet':
        raise ValueError(f'Unexpected NCBI EFetch root: {root.tag}')
    if root.find('.//ERROR') is not None:
        raise ValueError('NCBI EFetch returned an XML error instead of articles')
    results = {}
    for article in list(root.findall('PubmedArticle')) + list(root.findall('PubmedBookArticle')):
        pmid = (article.findtext('./MedlineCitation/PMID')
                or article.findtext('./BookDocument/PMID') or '').strip()
        if not pmid.isdigit():
            continue
        title_node = article.find('./MedlineCitation/Article/ArticleTitle')
        if title_node is None:
            title_node = article.find('./BookDocument/ArticleTitle')
        title = _xml_text(title_node)
        citation = article.find('./MedlineCitation')
        if citation is None:
            citation = article.find('./BookDocument')
        abstract_parts = []
        if citation is not None:
            for part in citation.findall('.//Abstract/AbstractText'):
                body = _xml_text(part)
                if not body:
                    continue
                label = (part.get('Label') or '').strip()
                if label:
                    abstract_parts.append('<h4>' + html.escape(label, quote=False)
                                          + '</h4>' + html.escape(body, quote=False))
                else:
                    abstract_parts.append(html.escape(body, quote=False))
        keywords = []
        if citation is not None:
            keywords = [_xml_text(k) for k in citation.findall('.//KeywordList/Keyword')]
        journal = _xml_text(article.find('./MedlineCitation/Article/Journal/Title'))
        if title or abstract_parts:
            results[pmid] = {
                'pmid': pmid, 'title': title,
                'abstractText': ' '.join(abstract_parts),
                'keywordList': {'keyword': [k for k in keywords if k]},
                'firstPublicationDate': _ncbi_date(article),
                'journalInfo': {'journal': {'title': journal or None}},
            }
    return results


def _read_zstd(path):
    with path.open('rb') as compressed, zstd.ZstdDecompressor().stream_reader(compressed) as stream:
        return stream.read()


def _iter_pubmed_cache(pubmed_dir):
    for path in sorted(pubmed_dir.glob('batch_*.json.zst')):
        records = json.loads(_read_zstd(path).decode('utf-8'))
        if not isinstance(records, list):
            raise ValueError(f'Expected a JSON list: {path}')
        for record in records:
            if not isinstance(record, dict):
                raise ValueError(f'Invalid PubMed record: {path}')
            yield path, 'europe_pmc', record
    for path in sorted(pubmed_dir.glob('ncbi_efetch_*.xml.zst')):
        for record in _parse_ncbi_xml(_read_zstd(path)).values():
            yield path, 'ncbi_efetch', record


def _cache_inventory(pubmed_dir, source):
    inventory, duplicate_records = {}, 0
    last_path, files_seen = None, 0
    for path, kind, record in _iter_pubmed_cache(pubmed_dir):
        if path != last_path:
            files_seen += 1
            last_path = path
            if files_seen == 1 or files_seen % 10 == 0:
                print(f'PubMed cache: reading file {files_seen}, '
                      f'{len(inventory):,} unique PMIDs so far: {path.name}', flush=True)
        pmid = str(record.get('pmid') or '').strip()
        if pmid not in source or not (record.get('title') or record.get('abstractText')):
            continue
        if pmid in inventory:
            duplicate_records += 1
            continue
        fetched_at = datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).replace(tzinfo=None)
        inventory[pmid] = (path.name, kind, fetched_at)
    return inventory, duplicate_records


def _ncbi_attempted_pmids(pubmed_dir):
    attempted = set()
    for path in pubmed_dir.glob('ncbi_efetch_*.request.json'):
        meta = json.loads(path.read_text(encoding='utf-8'))
        if not (pubmed_dir / meta['xml_file']).is_file():
            raise FileNotFoundError(f'NCBI request metadata has no XML: {path}')
        attempted.update(str(pmid) for pmid in meta['pmids'])
    return attempted


def _fetch_ncbi_missing(pubmed_dir, pmids, client=None):
    pending = sorted(set(pmids) - _ncbi_attempted_pmids(pubmed_dir), key=int)
    if len(pending) > 500:
        raise ValueError(f'{len(pending):,} PMID absent; expected at most 229. Check Drive/cache first.')
    print(f'NCBI EFetch: {len(pending):,} PMID(s) to request in '
          f'{(len(pending) + NCBI_BATCH_SIZE - 1) // NCBI_BATCH_SIZE} batch(es).', flush=True)
    own_client = client is None
    if own_client:
        client = httpx.Client(timeout=90.0, http2=True)
    fetched_batches = 0
    try:
        for start in range(0, len(pending), NCBI_BATCH_SIZE):
            chunk = pending[start:start + NCBI_BATCH_SIZE]
            print(f'NCBI EFetch: requesting batch {start // NCBI_BATCH_SIZE + 1}, '
                  f'{len(chunk):,} PMID(s)...', flush=True)
            digest = hashlib.sha1(','.join(chunk).encode('ascii')).hexdigest()[:12]
            stem = f'ncbi_efetch_{start // NCBI_BATCH_SIZE:03d}_{digest}'
            xml_file = pubmed_dir / f'{stem}.xml.zst'
            meta_file = pubmed_dir / f'{stem}.request.json'
            params = {'db': 'pubmed', 'id': ','.join(chunk), 'retmode': 'xml',
                      'tool': 'ViBioMIR'}
            api_key = os.environ.get('NCBI_API_KEY')
            if api_key:
                params['api_key'] = api_key
            for attempt, delay in enumerate((1, 3, 10, 30), 1):
                try:
                    response = client.get(NCBI_EFETCH_URL, params=params)
                    response.raise_for_status()
                    hits = _parse_ncbi_xml(response.content)
                    unexpected = set(hits) - set(chunk)
                    if unexpected:
                        raise ValueError(f'NCBI returned unrequested PMIDs: {sorted(unexpected)[:5]}')
                    break
                except (httpx.HTTPError, ET.ParseError) as exc:
                    if attempt == 4 or (isinstance(exc, httpx.HTTPStatusError)
                                        and exc.response.status_code in (401, 403)):
                        raise RuntimeError(f'NCBI EFetch failed; DB was not changed: {exc}') from exc
                    time.sleep(delay)
            raw = zstd.ZstdCompressor(level=6).compress(response.content)
            temp_xml = xml_file.with_suffix(xml_file.suffix + '.tmp')
            temp_xml.write_bytes(raw)
            os.replace(temp_xml, xml_file)
            meta = {'pmids': chunk, 'xml_file': xml_file.name,
                    'returned': sorted(hits, key=int),
                    'fetched_at': datetime.now(timezone.utc).isoformat()}
            temp_meta = meta_file.with_suffix(meta_file.suffix + '.tmp')
            temp_meta.write_text(json.dumps(meta, indent=2), encoding='utf-8')
            os.replace(temp_meta, meta_file)
            fetched_batches += 1
            print(f'NCBI EFetch batch {fetched_batches}: {len(hits):,}/{len(chunk):,} PMID; {xml_file.name}', flush=True)
            time.sleep(0.4 if not api_key else 0.12)
    finally:
        if own_client:
            client.close()
    return fetched_batches


def _sync_pubmed_status(status_db, pubmed_dir, source, inventory, attempted):
    wal = Path(str(status_db) + '.wal')
    if wal.exists():
        raise RuntimeError(f'DuckDB WAL exists; stop other DB writers first: {wal}')
    print('PubMed: backing up and updating status.duckdb...', flush=True)
    backup_dir = status_db.parent / 'backups'
    backup_dir.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    backup = backup_dir / f'{status_db.name}.before_ncbi_{stamp}'
    shutil.copy2(status_db, backup)
    matched = [(source[pmid][0], pmid, file, fetched_at,
                'application/xml' if kind == 'ncbi_efetch' else 'application/json')
               for pmid, (file, kind, fetched_at) in inventory.items()]
    missing = [(source[pmid][0], pmid) for pmid in sorted(attempted - set(inventory), key=int)
               if pmid in source and source[pmid][2] != 'ok']
    con = duckdb.connect(str(status_db))
    try:
        stage = pd.DataFrame(matched, columns=['id', 'pmid', 'batch_file', 'fetched_at', 'content_type'])
        con.register('_pubmed_found', stage)
        con.register('_pubmed_missing', pd.DataFrame(missing, columns=['id', 'pmid']))
        con.execute('BEGIN TRANSACTION')
        try:
            con.execute("""CREATE TABLE IF NOT EXISTS pubmed_cache_index (
                id BIGINT PRIMARY KEY, pmid VARCHAR, batch_file VARCHAR,
                fetched_at TIMESTAMP, imported_at TIMESTAMP)""")
            con.execute("""DELETE FROM pubmed_cache_index AS dst
                USING _pubmed_found AS src WHERE dst.id=src.id""")
            con.execute("""INSERT INTO pubmed_cache_index
                SELECT id,pmid,batch_file,fetched_at,current_timestamp FROM _pubmed_found""")
            con.execute("""UPDATE crawl_status AS dst SET
                status='ok', final_url=dst.url, http_code=NULL,
                content_type=src.content_type, n_bytes=NULL,
                fetched_at=src.fetched_at, shard=NULL, sha1_raw=NULL, error_msg=NULL,
                attempts=COALESCE(dst.attempts,0)+1
                FROM _pubmed_found AS src
                WHERE dst.id=src.id AND dst.status IS DISTINCT FROM 'ok'""")
            if missing:
                con.execute("""UPDATE crawl_status AS dst SET
                    status='http_error', http_code=NULL,
                    fetched_at=current_timestamp,
                    error_msg='NCBI EFetch returned no article for PMID',
                    attempts=COALESCE(dst.attempts,0)+1
                    FROM _pubmed_missing AS src
                    WHERE dst.id=src.id AND dst.status IS DISTINCT FROM 'ok'
                      AND (dst.status IS DISTINCT FROM 'http_error'
                        OR dst.error_msg IS DISTINCT FROM 'NCBI EFetch returned no article for PMID')""")
            confirmed = con.execute("""SELECT count(*) FROM crawl_status AS dst
                JOIN _pubmed_found AS src ON dst.id=src.id WHERE dst.status='ok'""").fetchone()[0]
            if confirmed != len(matched):
                raise RuntimeError(f'Only {confirmed:,}/{len(matched):,} cached PMIDs are ok')
            con.execute('COMMIT')
        except Exception:
            con.execute('ROLLBACK')
            raise
        con.execute('CHECKPOINT')
    finally:
        con.close()
    print(f'Status DB updated in place: {len(matched):,} ok, {len(missing):,} NCBI missing; backup: {backup}', flush=True)
    return backup


def _pubmed_doc_schema():
    block = pa.struct([('type', pa.string()), ('level', pa.int64()),
                       ('role', pa.string()), ('text', pa.string()),
                       ('char_start', pa.int64()), ('char_end', pa.int64())])
    return pa.schema([
        ('id', pa.int64()), ('url', pa.string()), ('final_url', pa.string()),
        ('lang', pa.string()), ('lang_conf', pa.float64()),
        ('title', pa.string()), ('published_date', pa.string()),
        ('site', pa.string()), ('category', pa.string()),
        ('text', pa.string()), ('blocks', pa.list_(block)),
        ('n_chars', pa.int64()), ('sha1_text', pa.string()),
        ('dup_group', pa.int64()), ('extractor', pa.string()),
        ('quality_flags', pa.list_(pa.string())), ('schema_version', pa.int64()),
    ])


def _check_pubmed_doc(doc):
    text = doc['text']
    blocks = doc['blocks']
    if doc['n_chars'] != len(text) or text != '\n\n'.join(block['text'] for block in blocks):
        raise ValueError(f'PubMed text/block mismatch: {doc["id"]}')
    if any(text[block['char_start']:block['char_end']] != block['text'] for block in blocks):
        raise ValueError(f'PubMed block offset mismatch: {doc["id"]}')


def _write_pubmed_docs(pubmed_dir, docs_dir, report_dir, source, cleaner, inventory):
    print(f'PubMed: extracting {len(source):,} docs into Parquet...', flush=True)
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S') + '_' + uuid.uuid4().hex[:8]
    output = docs_dir / f'pubmed_{run_id}'
    output.mkdir(parents=True, exist_ok=False)
    report_dir.mkdir(parents=True, exist_ok=True)
    schema = _pubmed_doc_schema()
    seen, duplicates, text_groups, files = set(), 0, {}, []
    counts = {'europe_pmc': 0, 'ncbi_efetch': 0, 'placeholder': 0}
    batch = []

    def flush():
        if not batch:
            return
        target = output / f'docs_{len(files):05d}.parquet'
        temp = target.with_suffix('.parquet.tmp')
        pq.write_table(pa.Table.from_pylist(batch, schema=schema), temp, compression='zstd')
        os.replace(temp, target)
        files.append(target.name)
        print(f'PubMed docs: {sum(counts.values()):,}/{len(source):,}; {target.name}', flush=True)
        batch.clear()

    for _path, kind, record in _iter_pubmed_cache(pubmed_dir):
        pmid = str(record.get('pmid') or '').strip()
        if (pmid not in inventory or inventory[pmid][0] != _path.name
                or not (record.get('title') or record.get('abstractText'))):
            continue
        if pmid in seen:
            duplicates += 1
            continue
        rid, url, _status = source[pmid]
        doc = cleaner(record, rid, url)
        if kind == 'ncbi_efetch':
            doc['extractor'] = 'pubmed_ncbi_efetch_v1'
        _check_pubmed_doc(doc)
        digest = doc['sha1_text']
        if doc['text']:
            doc['dup_group'] = text_groups.setdefault(digest, rid)
            if doc['dup_group'] == rid:
                doc['dup_group'] = None
        batch.append(doc)
        seen.add(pmid)
        counts[kind] += 1
        if len(batch) >= PUBMED_DOC_SHARD_SIZE:
            flush()

    missing = sorted(set(source) - seen, key=int)
    with (report_dir / 'pubmed_missing_after_ncbi.csv').open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.writer(handle)
        writer.writerow(['id', 'pmid', 'url'])
        for pmid in missing:
            rid, url, _status = source[pmid]
            writer.writerow([rid, pmid, url])
            batch.append({'id': rid, 'url': url, 'final_url': url,
                          'lang': 'en', 'lang_conf': 0.0, 'title': None,
                          'published_date': None, 'site': 'pubmed.ncbi.nlm.nih.gov',
                          'category': None, 'text': '', 'blocks': [], 'n_chars': 0,
                          'sha1_text': None, 'dup_group': None, 'extractor': 'placeholder',
                          'quality_flags': ['missing', 'ncbi_not_found'], 'schema_version': 2})
            counts['placeholder'] += 1
            if len(batch) >= PUBMED_DOC_SHARD_SIZE:
                flush()
    flush()
    if sum(counts.values()) != len(source) or len(seen) != len(inventory):
        raise RuntimeError('PubMed output coverage does not match cache/source inventory')
    report = {'expected_ids': len(source), 'content_docs': len(seen),
              'placeholder_docs': len(missing), 'europe_pmc_docs': counts['europe_pmc'],
              'ncbi_docs': counts['ncbi_efetch'], 'duplicate_cache_records': duplicates,
              'output': str(output), 'files': files}
    (output / 'manifest.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    (report_dir / 'pubmed_extraction_summary.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps({key: val for key, val in report.items() if key != 'files'},
                     ensure_ascii=False, indent=2), flush=True)
    return report


def run_pubmed_completion(status_db, pubmed_dir, docs_dir, report_dir, cleaner, client=None):
    """Fetch uncached PMIDs, sync the mounted Drive DB in place, and clean docs."""
    status_db, pubmed_dir = Path(status_db), Path(pubmed_dir)
    print(f'PubMed: starting; DB={status_db}; cache={pubmed_dir}', flush=True)
    if not status_db.is_file():
        raise FileNotFoundError(status_db)
    if not list(pubmed_dir.glob('batch_*.json.zst')):
        raise FileNotFoundError(f'No Europe PMC batch files: {pubmed_dir}')
    if Path(str(status_db) + '.wal').exists():
        raise RuntimeError('Stop other DuckDB writers before PubMed completion')
    print('PubMed: reading PMID IDs from status.duckdb...', flush=True)
    source = _pubmed_source_rows(status_db)
    print(f'PubMed: scanning cached batches for {len(source):,} PMID IDs...', flush=True)
    inventory, duplicates = _cache_inventory(pubmed_dir, source)
    missing = set(source) - set(inventory)
    print(f'PubMed DB={len(source):,}, cached={len(inventory):,}, missing={len(missing):,}, duplicates={duplicates:,}', flush=True)
    if len(missing) > 500:
        raise ValueError('Too many PubMed PMIDs are missing; check the mounted Drive account and cache')
    fetched_batches = _fetch_ncbi_missing(pubmed_dir, missing, client=client)
    inventory, duplicates = _cache_inventory(pubmed_dir, source)
    attempted = _ncbi_attempted_pmids(pubmed_dir)
    backup = _sync_pubmed_status(status_db, pubmed_dir, source, inventory, attempted)
    report = _write_pubmed_docs(pubmed_dir, Path(docs_dir), Path(report_dir),
                                source, cleaner, inventory)
    report.update(ncbi_new_batches=fetched_batches,
                  ncbi_attempted_pmids=len(attempted & set(source)),
                  status_backup=str(backup),
                  missing_report=str(Path(report_dir) / 'pubmed_missing_after_ncbi.csv'))
    summary = json.dumps(report, ensure_ascii=False, indent=2)
    (Path(report['output']) / 'manifest.json').write_text(summary, encoding='utf-8')
    (Path(report_dir) / 'pubmed_extraction_summary.json').write_text(summary, encoding='utf-8')
    return report


print('PubMed helper loaded.', flush=True)


# 4. Targeted NCBI retry
"""Retry only PubMed PMIDs still missing after the first NCBI pass.

Run after pubmed_completion_cell.py in the same Colab notebook.
"""


def _rescue_sidecar(pubmed_dir, stem, pmids):
    """Return cached hits for a completed request, or None if it needs sending."""
    xml_file = pubmed_dir / f'{stem}.xml.zst'
    meta_file = pubmed_dir / f'{stem}.request.json'
    if not meta_file.is_file():
        return None
    meta = json.loads(meta_file.read_text(encoding='utf-8'))
    if meta.get('pmids') != pmids or meta.get('xml_file') != xml_file.name:
        raise ValueError(f'NCBI retry sidecar does not match request: {meta_file}')
    if not xml_file.is_file():
        raise FileNotFoundError(f'NCBI retry XML missing: {xml_file}')
    hits = _parse_ncbi_xml(_read_zstd(xml_file))
    if sorted(hits, key=int) != meta.get('returned'):
        raise ValueError(f'NCBI retry sidecar/XML mismatch: {meta_file}')
    return set(hits)


def _rescue_request(pubmed_dir, stem, pmids, client, api_key):
    cached = _rescue_sidecar(pubmed_dir, stem, pmids)
    if cached is not None:
        print(f'NCBI retry cached: {stem}; {len(cached)}/{len(pmids)} found', flush=True)
        return cached
    params = {'db': 'pubmed', 'id': ','.join(pmids), 'retmode': 'xml',
              'tool': 'ViBioMIR'}
    if api_key:
        params['api_key'] = api_key
    for attempt, delay in enumerate((1, 3, 10, 30), 1):
        try:
            response = client.get(NCBI_EFETCH_URL, params=params)
            response.raise_for_status()
            hits = _parse_ncbi_xml(response.content)
            if set(hits) - set(pmids):
                raise ValueError(f'NCBI returned unrequested PMID(s): {set(hits) - set(pmids)}')
            break
        except (httpx.HTTPError, ET.ParseError, ValueError) as exc:
            if attempt == 4 or (isinstance(exc, httpx.HTTPStatusError)
                                and exc.response.status_code in (401, 403)):
                raise RuntimeError(f'NCBI retry failed for {stem}: {exc}') from exc
            print(f'NCBI retry error; waiting {delay}s before attempt {attempt + 1}: {exc}',
                  flush=True)
            time.sleep(delay)
    xml_file = pubmed_dir / f'{stem}.xml.zst'
    meta_file = pubmed_dir / f'{stem}.request.json'
    temp_xml = xml_file.with_suffix(xml_file.suffix + '.tmp')
    temp_xml.write_bytes(zstd.ZstdCompressor(level=6).compress(response.content))
    os.replace(temp_xml, xml_file)
    meta = {'pmids': pmids, 'xml_file': xml_file.name,
            'returned': sorted(hits, key=int),
            'fetched_at': datetime.now(timezone.utc).isoformat(),
            'reason': 'targeted NCBI retry after incomplete first pass'}
    temp_meta = meta_file.with_suffix(meta_file.suffix + '.tmp')
    temp_meta.write_text(json.dumps(meta, indent=2), encoding='utf-8')
    os.replace(temp_meta, meta_file)
    print(f'NCBI retry fetched: {stem}; {len(hits)}/{len(pmids)} found', flush=True)
    time.sleep(0.12 if api_key else 0.4)
    return set(hits)


def retry_ncbi_missing(pubmed_dir, pmids, report_dir, client=None, batch_size=10):
    """Retry missing PMIDs in small batches, then individually if still absent."""
    pubmed_dir, report_dir = Path(pubmed_dir), Path(report_dir)
    targets = sorted(set(str(pmid) for pmid in pmids), key=int)
    if not 1 <= batch_size <= 20:
        raise ValueError('batch_size must be from 1 to 20')
    if len(targets) > 500:
        raise ValueError(f'{len(targets)} non-ok PubMed IDs; check DB before retry')
    report_dir.mkdir(parents=True, exist_ok=True)
    print(f'NCBI targeted retry: {len(targets)} PMID(s), {batch_size} per batch', flush=True)
    api_key = os.environ.get('NCBI_API_KEY')
    own_client = client is None
    if own_client:
        client = httpx.Client(timeout=90.0, http2=True)
    batch_hits = set()
    single_hits = set()
    try:
        for start in range(0, len(targets), batch_size):
            chunk = targets[start:start + batch_size]
            digest = hashlib.sha1(','.join(chunk).encode('ascii')).hexdigest()[:12]
            stem = f'ncbi_efetch_retry_batch_{start // batch_size:03d}_{digest}'
            print(f'NCBI retry batch {start // batch_size + 1}: {len(chunk)} PMID(s)', flush=True)
            batch_hits.update(_rescue_request(pubmed_dir, stem, chunk, client, api_key))
        unresolved_after_batch = sorted(set(targets) - batch_hits, key=int)
        print(f'NCBI retry: {len(unresolved_after_batch)} PMID(s) need one-by-one check',
              flush=True)
        for index, pmid in enumerate(unresolved_after_batch, 1):
            print(f'NCBI retry single {index}/{len(unresolved_after_batch)}: {pmid}',
                  flush=True)
            single_hits.update(_rescue_request(
                pubmed_dir, f'ncbi_efetch_retry_single_{pmid}', [pmid], client, api_key))
    finally:
        if own_client:
            client.close()
    recovered = batch_hits | single_hits
    unresolved = sorted(set(targets) - recovered, key=int)
    result = {'target_pmids': len(targets), 'batch_hits': len(batch_hits),
              'single_hits': len(single_hits), 'recovered': len(recovered),
              'unresolved': len(unresolved), 'unresolved_pmids': unresolved,
              'report': str(report_dir / 'pubmed_targeted_retry.json')}
    Path(result['report']).write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(f'NCBI targeted retry complete: {len(recovered)}/{len(targets)} recovered; '
          f'{len(unresolved)} unresolved. Report: {result["report"]}', flush=True)
    return result

# 5. Retry only non-ok PubMed PMIDs and rebuild docs if any are recovered
if Path(str(STATUS_DB) + '.wal').exists():
    raise RuntimeError('Stop other processes writing status.duckdb before this retry')
source_rows = _pubmed_source_rows(STATUS_DB)
targets = sorted((pmid for pmid, row in source_rows.items() if row[2] != 'ok'), key=int)
print(f'PubMed remaining in status.duckdb: {len(targets)} PMID(s)', flush=True)
if targets:
    retry_result = retry_ncbi_missing(PUBMED_DIR, targets, REPORT_DIR, batch_size=10)
    if retry_result['recovered']:
        print('Recovered records found; rebuilding PubMed docs and syncing status.duckdb...',
              flush=True)
        result_pubmed = run_pubmed_completion(
            status_db=STATUS_DB,
            pubmed_dir=PUBMED_DIR,
            docs_dir=DOCS_DIR,
            report_dir=REPORT_DIR,
            cleaner=extract_pubmed_doc,
        )
        print('DONE. New PubMed docs:', result_pubmed['output'], flush=True)
        print('Content docs:', result_pubmed['content_docs'], flush=True)
        print('Still missing:', result_pubmed['placeholder_docs'], flush=True)
    else:
        print('No new records returned. Existing DB and docs remain as they were.', flush=True)
else:
    print('All PubMed IDs are already ok; nothing to retry.', flush=True)
